# 06. Vectores y embeddings: del conteo de palabras a BERT

En este cuaderno vamos a ver cómo una computadora puede representar el **significado** de las palabras con números.

La idea de fondo viene de la lingüística estructural. Es la **hipótesis distribucional**: palabras que aparecen en contextos parecidos tienden a tener significados parecidos (Harris, 1954). Firth (1957) lo resumió así: *"You shall know a word by the company it keeps"*.

Vamos a recorrer tres maneras de convertir esa idea en vectores:

1. Contar palabras en documentos (matriz término-documento)
2. Medir similitud entre vectores (coseno)
3. Embeddings estáticos con fastText
4. Analogías: $\mathbf{rey} - \mathbf{hombre} + \mathbf{mujer}$
5. Sesgos en los embeddings
6. Visualizar el espacio vectorial
7. El límite de los embeddings estáticos: un vector por palabra
8. Vectores contextuales con BERT

> Este cuaderno adapta y traduce al español ejercicios de la *Linguistic Data Science Worksheet* (jueves) de John McCrae, impartida en ESSLLI 2025.

## 0. Preparación

Si no tienes las bibliotecas instaladas, ejecuta en la terminal (o en una celda, anteponiendo `!`):

```bash
pip install numpy pandas matplotlib scikit-learn gensim requests transformers torch
```

`torch` y `transformers` sólo se usan en la sección 8. Son pesadas, así que puedes instalarlas al llegar ahí. Las versiones recientes de `transformers` necesitan PyTorch 2.6 o más. Si ves el aviso *Disabling PyTorch because PyTorch >= ... is required*, actualiza las dos y reinicia el kernel:

```bash
pip install -U torch transformers
```

Este notebook debe ejecutarse desde la raíz del repositorio para que funcionen las rutas.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATOS = Path("datos/embeddings")
DATOS.mkdir(parents=True, exist_ok=True)

---
# Parte 1: Palabras como vectores

## 1. La matriz término-documento

La forma más sencilla de representar una palabra con números es **contar** en qué documentos aparece.

Una **matriz término-documento** tiene una fila por palabra y una columna por documento. Cada celda dice cuántas veces aparece la palabra en ese documento. La fila de una palabra es su **vector**.

Usaremos un corpus pequeño, escrito para este ejercicio, con nueve documentos sobre tres temas: cocina, pesca e internet.

In [ ]:
documentos = {
    "cocina_1": "Para la cena compramos pan, queso y vino. El pan estaba recién horneado y el vino era tinto.",
    "cocina_2": "La receta lleva pescado, ajo y aceite. Sirve el pescado con pan y una copa de vino blanco.",
    "cocina_3": "En la panadería venden pan dulce. Por la tarde tomamos café con pan y queso.",
    "pesca_1": "Los pescadores salieron en el barco al amanecer. Echaron la red al mar y sacaron mucho pescado.",
    "pesca_2": "La red del barco se rompió con las olas. En el puerto reparan la red y limpian el pescado.",
    "pesca_3": "El mar estaba tranquilo. El barco volvió al puerto con la red llena.",
    "internet_1": "Me conecté a la red del café para revisar mi correo. La red era lenta y no cargaba la página.",
    "internet_2": "Publiqué una foto en la red social y mis amigos comentaron. Después contesté el correo.",
    "internet_3": "La página no abre porque falla la conexión a internet. Reinicia el módem y revisa la red.",
}

Primero **tokenizamos**: pasamos todo a minúsculas y separamos las palabras con una expresión regular. `\w+` captura secuencias de letras y números, incluidas las letras con acento.

In [ ]:
def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())

tokenizar(documentos["pesca_3"])

Ahora contamos las palabras de cada documento y armamos la matriz con pandas. Para que sea legible, nos quedamos con un vocabulario de palabras de contenido.

In [ ]:
vocabulario = ["pan", "vino", "queso", "pescado", "barco", "mar",
               "puerto", "red", "correo", "página", "internet", "café"]

conteos = {nombre: Counter(tokenizar(texto)) for nombre, texto in documentos.items()}

matriz = pd.DataFrame(
    {nombre: [conteo[palabra] for palabra in vocabulario] for nombre, conteo in conteos.items()},
    index=vocabulario,
)
matriz

Cada fila es el vector de una palabra. Por ejemplo, el vector de *barco* tiene nueve dimensiones, una por documento:

In [ ]:
matriz.loc["barco"].to_numpy()

**Pregunta.** Observa las filas de *pan*, *pescado* y *red*. ¿Qué palabras tienen filas parecidas? ¿Qué tiene de especial la fila de *red*?

## 2. Similitud coseno

Para comparar dos vectores usamos la **similitud coseno**. Mide el ángulo entre ellos, sin importar su longitud:

$$\cos(\mathbf{u},\mathbf{v}) = \frac{\mathbf{u}\cdot\mathbf{v}}{\lVert\mathbf{u}\rVert\,\lVert\mathbf{v}\rVert}$$

- $\mathbf{u}\cdot\mathbf{v}$ es el **producto punto**: multiplicamos componente por componente y sumamos.
- $\lVert\mathbf{u}\rVert$ es la **norma** (longitud) del vector.

El resultado va de $-1$ a $1$. Con conteos nunca hay valores negativos, así que aquí va de $0$ (nada en común) a $1$ (misma dirección).

¿Por qué no usar la distancia euclidiana? Porque depende de la frecuencia. Una palabra muy frecuente tiene un vector largo y queda lejos de todas, aunque aparezca en los mismos contextos. El coseno sólo mira la dirección.

In [ ]:
def coseno(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

pan = matriz.loc["pan"].to_numpy()
vino = matriz.loc["vino"].to_numpy()
barco = matriz.loc["barco"].to_numpy()

print("pan - vino: ", round(coseno(pan, vino), 3))
print("pan - barco:", round(coseno(pan, barco), 3))

scikit-learn tiene una función que calcula el coseno entre **todas** las filas de una matriz a la vez. El resultado es una matriz de similitud palabra-palabra.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes = pd.DataFrame(cosine_similarity(matriz), index=vocabulario, columns=vocabulario)
similitudes.round(2)

Es más fácil leerla como un **mapa de calor**:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
imagen = ax.imshow(similitudes, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(vocabulario)), vocabulario, rotation=45, ha="right")
ax.set_yticks(range(len(vocabulario)), vocabulario)
fig.colorbar(imagen, label="similitud coseno")
ax.set_title("Similitud entre palabras (matriz término-documento)")
plt.tight_layout()
plt.show()

### ✅ Ejercicio 1 (resuelto)

1. Escribe una función `mas_similares(palabra, n=3)` que regrese las `n` palabras más parecidas a `palabra`, sin contarse a sí misma.
2. Úsala con *pescado* y con *red*. ¿Qué observas en los vecinos de *red*?
3. Con `euclidean_distances`, calcula la distancia de *barco* a *red* y a *queso*. Compáralas con el coseno. ¿Coincide el orden?

In [ ]:
# 1.
def mas_similares(palabra, n=3):
    return similitudes[palabra].drop(palabra).sort_values(ascending=False).head(n)

# 2.
print(mas_similares("pescado"), "\n")
print(mas_similares("red"))

In [ ]:
# 3.
from sklearn.metrics.pairwise import euclidean_distances

distancias = pd.DataFrame(euclidean_distances(matriz), index=vocabulario, columns=vocabulario)
for palabra in ["red", "queso"]:
    print(f"barco - {palabra:<6} euclidiana: {distancias.loc['barco', palabra]:.2f}   coseno: {similitudes.loc['barco', palabra]:.2f}")

matriz.sum(axis=1)  # frecuencia total de cada palabra

**2.** Los vecinos de *pescado* son *barco* y *vino*, porque aparece en documentos de pesca y de cocina. Los de *red* son *barco* y *puerto*, pero también *página*. Su vector **mezcla dos sentidos**: la red de pesca y la red informática. Volveremos a este problema en la sección 7.

**3.** No coincide. En distancia euclidiana, *barco* queda más cerca de *queso* que de *red*, aunque *queso* nunca aparece con *barco*. La razón es la frecuencia: *red* aparece 8 veces y su vector es largo, así que queda lejos de casi todo. El coseno sólo mira la dirección y pone a *red* mucho más cerca de *barco*.

### El problema de la dispersión

En un corpus real, el vocabulario tiene cientos de miles de palabras y puede haber millones de documentos. La matriz es enorme y casi todas sus celdas son cero: es una matriz **dispersa** (*sparse*).

Los **embeddings** resuelven esto. En lugar de una dimensión por documento, cada palabra se representa con un vector **denso** de unas cuantas centenas de números reales, aprendido a partir del contexto.

---
# Parte 2: Embeddings estáticos

## 3. fastText

Modelos como **word2vec** (Mikolov et al., 2013) y **fastText** (Bojanowski et al., 2017) aprenden vectores con una tarea de predicción: dada una palabra, adivinar qué palabras aparecen a su alrededor (o al revés). Las palabras que comparten contextos terminan con vectores cercanos.

fastText además aprende vectores para fragmentos de palabras (*n-gramas de caracteres*). Eso ayuda con la morfología del español: *cantábamos* comparte piezas con *cantar* y *cantaba*.

Usaremos los vectores de fastText para el español entrenados con Common Crawl y Wikipedia (Grave et al., 2018). Tienen 300 dimensiones. Se distribuyen con licencia CC BY-SA 3.0.

El archivo completo pesa 1.3 GB comprimido y tiene dos millones de palabras, ordenadas de la más frecuente a la menos frecuente. Sólo necesitamos las primeras 100 000. La siguiente función descarga el archivo **por partes** y se detiene en cuanto tiene suficientes palabras. Son unos 200 MB y se guardan en `datos/embeddings/`. Si el archivo ya existe, no se descarga de nuevo.

In [ ]:
import gzip
import requests

URL_VECTORES = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz"
N_PALABRAS = 100_000
RUTA_VECTORES = DATOS / f"cc.es.300.{N_PALABRAS // 1000}k.vec"


def descargar_vectores(url, ruta, n_palabras):
    # Descarga sólo las primeras n_palabras del archivo de fastText
    if ruta.exists():
        print(f"Ya existe {ruta}. No se descarga de nuevo.")
        return
    temporal = ruta.with_suffix(".parcial")
    with requests.get(url, stream=True, timeout=60) as respuesta:
        respuesta.raise_for_status()
        with gzip.open(respuesta.raw, mode="rt", encoding="utf-8") as entrada, \
             open(temporal, "w", encoding="utf-8") as salida:
            # La primera línea dice cuántas palabras y cuántas dimensiones hay
            _, dimensiones = entrada.readline().split()
            salida.write(f"{n_palabras} {dimensiones}\n")
            for i, linea in enumerate(entrada):
                if i >= n_palabras:
                    break
                salida.write(linea)
    temporal.rename(ruta)
    print(f"Guardado en {ruta}")


descargar_vectores(URL_VECTORES, RUTA_VECTORES, N_PALABRAS)

Cargamos los vectores con **gensim**. El objeto `KeyedVectors` funciona como un diccionario: la llave es la palabra y el valor es su vector.

In [ ]:
from gensim.models import KeyedVectors

vectores = KeyedVectors.load_word2vec_format(RUTA_VECTORES, binary=False)
print(len(vectores), "palabras,", vectores.vector_size, "dimensiones")

In [ ]:
vectores["perro"][:10]  # las primeras 10 de 300 dimensiones

Los números de cada dimensión no tienen una interpretación directa. Lo que importa es la **posición relativa** de los vectores.

Los vectores de fastText distinguen mayúsculas: *Lima* (la ciudad) y *lima* (la fruta o la herramienta) son entradas distintas.

Como sólo cargamos 100 000 palabras, algunas pueden faltar. Esta función auxiliar quita de una lista las palabras que no están y avisa cuáles fueron:

In [ ]:
def en_vocabulario(palabras):
    faltan = [p for p in palabras if p not in vectores.key_to_index]
    if faltan:
        print("No están en el vocabulario:", faltan)
    return [p for p in palabras if p in vectores.key_to_index]

`similarity()` calcula el coseno entre dos palabras y `most_similar()` busca los vecinos más cercanos.

In [ ]:
print("pan - vino: ", round(float(vectores.similarity("pan", "vino")), 3))
print("pan - barco:", round(float(vectores.similarity("pan", "barco")), 3))

In [ ]:
vectores.most_similar("perro", topn=10)

### ✅ Ejercicio 2 (resuelto)

1. Busca los 10 vecinos más cercanos de *red*. ¿Qué sentido domina?
2. Haz lo mismo con *banco* y con *gato*. ¿Aparecen palabras de más de un sentido?
3. Compara `similarity("banco", "dinero")` con `similarity("banco", "sentarse")`.

In [ ]:
# 1.
vectores.most_similar("red", topn=10)

In [ ]:
# 2.
for palabra in ["banco", "gato"]:
    vecinos = [vecino for vecino, _ in vectores.most_similar(palabra, topn=10)]
    print(palabra, "->", ", ".join(vecinos))

In [ ]:
# 3.
print("banco - dinero:  ", round(float(vectores.similarity("banco", "dinero")), 3))
print("banco - sentarse:", round(float(vectores.similarity("banco", "sentarse")), 3))

En un corpus web como Common Crawl, el sentido de *red* como 'internet' o 'red social' es mucho más frecuente que el de pesca. El vector de *red* queda cerca de ese sentido. Con *banco* pasa algo parecido con el sentido financiero. Un embedding estático tiene **un solo vector por palabra**, y ese vector queda dominado por el sentido más frecuente en el corpus de entrenamiento.

> **Para pensar: cambio semántico.** Hamilton, Leskovec y Jurafsky (2016) entrenaron un espacio vectorial por cada década del inglés de 1800 a 1990. Así pudieron seguir cómo cambian los vecinos de una palabra. *Gay* pasa de estar cerca de *cheerful* y *frolicsome* a estar cerca de *lesbian* y *homosexual*. En español podríamos esperar algo parecido con *red*: en un corpus del siglo XIX sus vecinos serían *pesca*, *anzuelo* o *barca*, y en uno actual, *internet* o *conexión*. Para comparar dos espacios entrenados por separado hay que alinearlos primero. Ellos usaron una técnica llamada Procrustes ortogonal.

## 4. Analogías: rey − hombre + mujer

Una propiedad llamativa de estos espacios es que algunas relaciones entre palabras se vuelven **direcciones**. La diferencia entre *rey* y *hombre* se parece a la diferencia entre *reina* y *mujer*:

$$\mathbf{rey} - \mathbf{hombre} \approx \mathbf{reina} - \mathbf{mujer}$$

Si despejamos *reina*, podemos resolver la analogía "*hombre* es a *rey* como *mujer* es a ¿?":

$$\mathbf{rey} - \mathbf{hombre} + \mathbf{mujer} \approx \ ?$$

En gensim, las palabras que se suman van en `positive` y las que se restan en `negative`:

In [ ]:
def analogia(a, b, c, n=1):
    # a es a b como c es a ¿?  ->  b - a + c
    return vectores.most_similar(positive=[b, c], negative=[a], topn=n)

analogia("hombre", "rey", "mujer", n=5)

### ¿Funciona tan bien como parece?

`most_similar()` hace una trampa discreta: **nunca regresa las palabras de entrada**. Calculemos la operación a mano con numpy para ver qué pasa sin esa regla.

Usamos vectores **normalizados** (de longitud 1), igual que gensim por dentro. Luego `similar_by_vector()` busca las palabras más cercanas a un vector cualquiera, sin excluir nada.

In [ ]:
def v(palabra):
    return vectores.get_vector(palabra, norm=True)

resultado = v("rey") - v("hombre") + v("mujer")
vectores.similar_by_vector(resultado, topn=5)

Muchas veces la palabra más cercana al resultado es el propio *rey*. *Reina* está cerca, pero no siempre en primer lugar. Nissim, van Noord y van der Goot (2020) discuten este problema en *Fair is Better than Sensational*. Muchos ejemplos famosos de analogías dependen de esa exclusión.

La lección es metodológica: el resultado depende del espacio vectorial **y** de cómo está programado el método que lo consulta.

### ✅ Ejercicio 3 (resuelto)

Prueba estas analogías con `analogia()` y con el cálculo a mano.

1. Morfológica: *comer* es a *comió* como *caminar* es a ¿?
2. Número: *perro* es a *perros* como *gato* es a ¿?
3. Geográfica: *Perú* es a *Lima* como *Chile* es a ¿?
4. Inventa una analogía que **falle**. ¿Por qué crees que falla?

In [ ]:
pruebas = [
    ("comer", "comió", "caminar"),
    ("perro", "perros", "gato"),
    ("Perú", "Lima", "Chile"),
]

for a, b, c in pruebas:
    if len(en_vocabulario([a, b, c])) < 3:
        continue
    con_exclusion = analogia(a, b, c)[0][0]
    sin_exclusion = vectores.similar_by_vector(v(b) - v(a) + v(c), topn=1)[0][0]
    print(f"{a} : {b} :: {c} : {con_exclusion:<12} (a mano: {sin_exclusion})")

Las analogías morfológicas suelen funcionar bien. La flexión del español es muy regular y deja huellas claras en los contextos: *comió* y *caminó* aparecen en posiciones sintácticas parecidas. Las analogías que fallan suelen involucrar palabras poco frecuentes, polisémicas o relaciones que el corpus casi no expresa.

## 5. Sesgos en los embeddings

Los embeddings aprenden de textos escritos por personas. También aprenden sus **estereotipos**. Bolukbasi et al. (2016) lo mostraron en inglés con la analogía *man : computer programmer :: woman : homemaker*.

En español hay una complicación interesante: el **género gramatical**. Si preguntamos *hombre : médico :: mujer : ¿?*, la respuesta esperable es *médica*. Eso no es un sesgo, es morfología.

In [ ]:
analogia("hombre", "médico", "mujer", n=5)

Para separar el sesgo de la gramática conviene usar sustantivos **epicenos**, que tienen la misma forma para cualquier género: *periodista*, *dentista*, *astronauta*.

Definimos una **dirección de género** como la diferencia $\mathbf{hombre} - \mathbf{mujer}$. Después medimos el coseno de cada profesión con esa dirección. Un valor positivo indica que la palabra está más cerca de *hombre*. Un valor negativo indica que está más cerca de *mujer*.

In [ ]:
direccion = v("hombre") - v("mujer")

profesiones = en_vocabulario([
    "astronauta", "electricista", "futbolista", "economista", "periodista",
    "dentista", "pianista", "recepcionista", "modelo", "oficinista",
])

inclinacion = pd.Series(
    {p: coseno(v(p), direccion) for p in profesiones}
).sort_values()

inclinacion.plot.barh(figsize=(6, 4), color=["tab:orange" if x < 0 else "tab:blue" for x in inclinacion])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("← más cerca de 'mujer'        más cerca de 'hombre' →")
plt.title("Profesiones epicenas en la dirección hombre − mujer")
plt.tight_layout()
plt.show()

**Preguntas.**

- ¿Qué profesiones quedan de cada lado? ¿Corresponde con estereotipos que reconoces?
- El modelo no "cree" nada. ¿De dónde viene entonces esta inclinación?
- Si usaras estos vectores en un sistema que filtra currículos, ¿qué consecuencias podría tener?

## 6. Visualizar el espacio vectorial

No podemos ver 300 dimensiones. Para dibujar los vectores tenemos que **reducirlos a dos**. Hay dos métodos comunes:

- **PCA** (análisis de componentes principales) busca las dos direcciones en las que los datos varían más y proyecta todo sobre ellas. Es lineal y siempre da el mismo resultado.
- **t-SNE** intenta conservar quién es vecino de quién. Agrupa mejor, pero las distancias entre grupos no significan mucho y el resultado cambia con sus parámetros.

In [ ]:
from sklearn.decomposition import PCA


def graficar_palabras(grupos, metodo="pca"):
    # grupos: diccionario {nombre del grupo: lista de palabras}
    palabras, etiquetas = [], []
    for grupo, lista in grupos.items():
        for palabra in en_vocabulario(lista):
            palabras.append(palabra)
            etiquetas.append(grupo)

    matriz_vectores = np.array([vectores[p] for p in palabras])

    if metodo == "pca":
        coordenadas = PCA(n_components=2).fit_transform(matriz_vectores)
    else:
        from sklearn.manifold import TSNE
        coordenadas = TSNE(n_components=2, perplexity=min(len(palabras) - 1, 10),
                           init="pca", random_state=23).fit_transform(matriz_vectores)

    plt.figure(figsize=(8, 7))
    for grupo in grupos:
        indices = [i for i, e in enumerate(etiquetas) if e == grupo]
        plt.scatter(coordenadas[indices, 0], coordenadas[indices, 1], label=grupo)
    for (x, y), palabra in zip(coordenadas, palabras):
        plt.annotate(palabra, (x, y), xytext=(4, 2), textcoords="offset points")
    plt.legend()
    plt.title(f"Palabras proyectadas con {metodo.upper()}")
    plt.show()


grupos = {
    "animales": ["perro", "gato", "caballo", "vaca", "pájaro", "ratón"],
    "bebidas": ["vino", "cerveza", "tequila", "mezcal", "café", "agua"],
    "países": ["México", "Perú", "Chile", "España", "Francia", "Japón"],
    "verbos": ["comer", "beber", "correr", "caminar", "dormir", "leer"],
}

graficar_palabras(grupos, metodo="pca")

### ✅ Ejercicio 4 (resuelto)

1. Repite la gráfica con `metodo="tsne"`. ¿Qué diferencias ves con PCA?
2. ¿Dónde queda *ratón*? Agrega *teclado*, *pantalla* y *computadora* a un grupo nuevo. ¿Se mueve *ratón* hacia ellas?

In [ ]:
# 1.
graficar_palabras(grupos, metodo="tsne")

In [ ]:
# 2.
grupos_con_tecnologia = {**grupos, "tecnología": ["teclado", "pantalla", "computadora"]}
graficar_palabras(grupos_con_tecnologia, metodo="pca")

**1.** t-SNE separa más los grupos, pero la distancia entre un grupo y otro no es interpretable. En PCA las distancias globales se conservan mejor, aunque los grupos pueden encimarse.

**2.** *Ratón* queda en un punto intermedio o se acerca a *teclado* y *pantalla*. En PCA la proyección depende de todas las palabras graficadas, así que agregar un grupo puede mover a todos. Pero el vector de *ratón* es el mismo: un solo punto para el animal y para el dispositivo.

---
# Parte 3: Vectores contextuales

## 7. El límite: un vector por palabra

Todo lo que vimos hasta ahora tiene el mismo problema. Hay **un solo vector por palabra** (por *tipo*), sin importar en qué oración aparezca:

- *Me senté en un **banco** del parque.*
- *Abrí una cuenta en el **banco**.*

Para fastText, las dos ocurrencias de *banco* son idénticas:

In [ ]:
banco_parque = vectores["banco"]
banco_cuenta = vectores["banco"]
coseno(banco_parque, banco_cuenta)

Desde la lingüística esto es insatisfactorio. Sabemos que el sentido se resuelve **en contexto**. Los modelos de lenguaje basados en *transformers* hacen justamente eso.

## 8. BERT: un vector por ocurrencia

**BERT** (Devlin et al., 2019) es un modelo de lenguaje que lee la oración completa y calcula un vector para **cada ocurrencia** (cada *token*) tomando en cuenta todas las demás palabras. A estos vectores se les llama **contextuales**.

**El objetivo de esta sección** es comprobar que las ocurrencias de una palabra ambigua se agrupan según su sentido, aunque nunca le digamos al modelo cuáles son los sentidos. Eso es algo que ningún embedding estático puede hacer.

Usaremos **BETO** (Cañete et al., 2020), un BERT entrenado con texto en español por la Universidad de Chile. La primera vez se descargan unos 440 MB.

In [ ]:
# !pip install transformers torch
import torch
import transformers
from transformers import AutoTokenizer, AutoModel

MODELO_BERT = "dccuchile/bert-base-spanish-wwm-uncased"

tokenizador = AutoTokenizer.from_pretrained(MODELO_BERT)
modelo = AutoModel.from_pretrained(MODELO_BERT)
modelo.eval();  # modo de uso, no de entrenamiento

print("torch", torch.__version__, "| transformers", transformers.__version__)

### 8.1 Tokenización en subpalabras

BERT no trabaja con palabras completas sino con **subpalabras** (*WordPiece*). Las palabras frecuentes son un solo token. Las raras se parten en pedazos, y los pedazos que continúan una palabra empiezan con `##`. Además se agregan dos tokens especiales: `[CLS]` al inicio y `[SEP]` al final.

In [ ]:
codificado = tokenizador("Me senté en un banco del parque y leí un libro larguísimo.")
print(codificado["input_ids"])
print(tokenizador.convert_ids_to_tokens(codificado["input_ids"]))

### 8.2 Capas

BERT procesa la oración en **capas**. Con `output_hidden_states=True` obtenemos la representación de cada token en cada capa. BETO tiene 12 capas más una capa inicial (la 0), que es sólo el vector de cada token antes de mirar el contexto.

El resultado de cada capa es un tensor con forma `(oraciones, tokens, dimensiones)`.

In [ ]:
entrada = tokenizador("Me senté en un banco del parque.", return_tensors="pt")

with torch.no_grad():  # no vamos a entrenar, así que no guardamos gradientes
    salida = modelo(**entrada, output_hidden_states=True)

print("Número de capas:", len(salida.hidden_states))
print("Forma de cada capa:", tuple(salida.hidden_states[-1].shape))

### 8.3 El vector de una palabra en contexto

Esta función regresa el vector de una palabra dentro de una oración, en la capa que elijamos. Si la palabra se partió en varias subpalabras, promedia sus vectores.

Para encontrar la palabra usa `word_ids()`, que dice a qué palabra de la oración pertenece cada token.

In [ ]:
def vector_contextual(oracion, palabra, capa=10):
    entrada = tokenizador(oracion, return_tensors="pt")
    with torch.no_grad():
        salida = modelo(**entrada, output_hidden_states=True)
    estados = salida.hidden_states[capa][0]  # (tokens, dimensiones)

    # Agrupamos los tokens por palabra y buscamos la que coincide
    for id_palabra in sorted(set(i for i in entrada.word_ids() if i is not None)):
        inicio, fin = entrada.word_to_chars(id_palabra)
        if oracion[inicio:fin].lower() == palabra.lower():
            posiciones = [i for i, w in enumerate(entrada.word_ids()) if w == id_palabra]
            return estados[posiciones].mean(dim=0).numpy()

    raise ValueError(f"'{palabra}' no aparece en: {oracion}")


v1 = vector_contextual("Me senté en un banco del parque.", "banco")
v2 = vector_contextual("Abrí una cuenta en el banco.", "banco")
print("Dimensiones:", v1.shape)
print("Coseno entre los dos 'banco':", round(coseno(v1, v2), 3))

A diferencia de fastText, los dos vectores de *banco* ya no son idénticos. Ahora veamos si se agrupan por sentido con más ejemplos.

In [ ]:
oraciones_banco = {
    "asiento": [
        "Me senté en un banco del parque a leer.",
        "El banco de madera estaba recién pintado.",
        "Los abuelos platican en el banco de la plaza.",
        "Dejé la mochila sobre el banco del jardín.",
        "En la iglesia no quedaba ningún banco libre.",
        "El carpintero hizo un banco para la cocina.",
    ],
    "institución": [
        "Abrí una cuenta en el banco la semana pasada.",
        "El banco aprobó mi préstamo para la casa.",
        "Tengo que ir al banco a depositar el cheque.",
        "El banco central subió las tasas de interés.",
        "Mi tarjeta del banco ya está vencida.",
        "Trabaja como cajera en un banco del centro.",
    ],
}


def vectores_por_sentido(oraciones_por_sentido, palabra, capa=10):
    filas, sentidos, textos = [], [], []
    for sentido, oraciones in oraciones_por_sentido.items():
        for oracion in oraciones:
            filas.append(vector_contextual(oracion, palabra, capa))
            sentidos.append(sentido)
            textos.append(oracion)
    return np.array(filas), sentidos, textos


vec_banco, sentidos_banco, textos_banco = vectores_por_sentido(oraciones_banco, "banco")
vec_banco.shape

Proyectamos los 12 vectores con PCA y coloreamos cada punto según el sentido que **nosotros** sabemos que tiene. El modelo no recibió esa información.

In [ ]:
def graficar_sentidos(matriz_vectores, sentidos, textos, titulo):
    coordenadas = PCA(n_components=2).fit_transform(matriz_vectores)
    plt.figure(figsize=(9, 6))
    for sentido in dict.fromkeys(sentidos):
        indices = [i for i, s in enumerate(sentidos) if s == sentido]
        plt.scatter(coordenadas[indices, 0], coordenadas[indices, 1], label=sentido, s=60)
    for (x, y), texto in zip(coordenadas, textos):
        corto = texto if len(texto) < 35 else texto[:32] + "..."
        plt.annotate(corto, (x, y), xytext=(4, 2), textcoords="offset points", fontsize=8)
    plt.legend()
    plt.title(titulo)
    plt.show()


graficar_sentidos(vec_banco, sentidos_banco, textos_banco, "Ocurrencias de 'banco' en BETO (capa 10)")

También podemos ver la matriz de cosenos entre todas las ocurrencias. Si el modelo distingue los sentidos, aparecen dos **bloques**: similitud alta dentro de cada sentido y más baja entre sentidos.

In [ ]:
def mapa_similitud(matriz_vectores, sentidos, titulo):
    sim = cosine_similarity(matriz_vectores)
    plt.figure(figsize=(6, 5))
    plt.imshow(sim, cmap="viridis")
    plt.colorbar(label="similitud coseno")
    etiquetas = [f"{s[:5]}{i}" for i, s in enumerate(sentidos)]
    plt.xticks(range(len(etiquetas)), etiquetas, rotation=90, fontsize=7)
    plt.yticks(range(len(etiquetas)), etiquetas, fontsize=7)
    plt.title(titulo)
    plt.tight_layout()
    plt.show()


def similitud_intra_inter(matriz_vectores, sentidos):
    # Promedio del coseno entre ocurrencias del mismo sentido y de sentidos distintos
    sim = cosine_similarity(matriz_vectores)
    sentidos = np.array(sentidos)
    mismo = sentidos[:, None] == sentidos[None, :]
    fuera_diagonal = ~np.eye(len(sentidos), dtype=bool)
    return sim[mismo & fuera_diagonal].mean(), sim[~mismo].mean()


mapa_similitud(vec_banco, sentidos_banco, "Coseno entre ocurrencias de 'banco'")

intra, inter = similitud_intra_inter(vec_banco, sentidos_banco)
print(f"Mismo sentido:     {intra:.3f}")
print(f"Distinto sentido:  {inter:.3f}")

### ✅ Ejercicio 5 (resuelto)

¿En qué capa se distinguen mejor los sentidos? Calcula la diferencia entre la similitud intra-sentido y la inter-sentido para todas las capas, de la 0 a la 12, y grafícala.

¿Qué pasa en la capa 0? ¿Por qué?

In [ ]:
diferencias = []
for capa in range(len(salida.hidden_states)):
    matriz_capa, sentidos_capa, _ = vectores_por_sentido(oraciones_banco, "banco", capa=capa)
    intra, inter = similitud_intra_inter(matriz_capa, sentidos_capa)
    diferencias.append(intra - inter)

plt.plot(range(len(diferencias)), diferencias, marker="o")
plt.xlabel("capa")
plt.ylabel("similitud intra − inter")
plt.title("¿En qué capa se separan los sentidos de 'banco'?")
plt.show()

En la **capa 0** la diferencia es casi nula. Ahí cada token sólo tiene su vector de vocabulario más un vector de posición: el modelo todavía no ha mirado el contexto. Es lo más parecido a un embedding estático. La separación crece en las capas intermedias y altas, donde el modelo ya integró la información de la oración. Suele ser menor en la última capa, que está más especializada en la tarea con la que se entrenó BERT (predecir palabras ocultas).

### ✅ Ejercicio 6 (resuelto)

Repite el análisis con *red*, usando oraciones del sentido 'pesca' y del sentido 'internet'. Es la misma palabra cuyo vector de la sección 1 mezclaba dos temas.

In [ ]:
oraciones_red = {
    "pesca": [
        "Los pescadores echaron la red al mar.",
        "La red se llenó de sardinas.",
        "El viejo remendaba la red en el muelle.",
        "Sacaron la red con tres peces enormes.",
        "La red del barco se enredó en las rocas.",
        "Compró una red nueva para pescar camarón.",
    ],
    "internet": [
        "No me puedo conectar a la red de la oficina.",
        "La red está muy lenta hoy.",
        "Publicó el video en una red social.",
        "Cambia la contraseña de la red inalámbrica.",
        "La red de la escuela bloquea esa página.",
        "Se cayó la red y perdimos la videollamada.",
    ],
}

vec_red, sentidos_red, textos_red = vectores_por_sentido(oraciones_red, "red")
graficar_sentidos(vec_red, sentidos_red, textos_red, "Ocurrencias de 'red' en BETO (capa 10)")

intra, inter = similitud_intra_inter(vec_red, sentidos_red)
print(f"Mismo sentido:     {intra:.3f}")
print(f"Distinto sentido:  {inter:.3f}")

**Para seguir explorando**

- Prueba con *gato* ('animal' y 'herramienta para levantar el coche') o *vela* ('de cera' y 'de barco').
- ¿Qué pasa con una oración ambigua de verdad, como *Lo esperé en el banco*? ¿Dónde cae su punto?
- Los grandes modelos de lenguaje actuales también trabajan con vectores contextuales como estos, sólo que con muchas más capas y dimensiones.

---
## Chuleta

| Tarea | Código |
|---|---|
| Tokenizar con regex | `re.findall(r"\w+", texto.lower())` |
| Similitud coseno | `cosine_similarity(matriz)` (sklearn) |
| Cargar vectores | `KeyedVectors.load_word2vec_format(ruta)` (gensim) |
| Vector de una palabra | `vectores["palabra"]` |
| Vector normalizado | `vectores.get_vector("palabra", norm=True)` |
| Coseno entre dos palabras | `vectores.similarity("a", "b")` |
| Vecinos más cercanos | `vectores.most_similar("palabra", topn=10)` |
| Analogía b − a + c | `vectores.most_similar(positive=[b, c], negative=[a])` |
| Vecinos de un vector | `vectores.similar_by_vector(vector, topn=5)` |
| Reducir a 2 dimensiones | `PCA(n_components=2).fit_transform(X)` |
| Cargar BERT | `AutoTokenizer.from_pretrained(m)`, `AutoModel.from_pretrained(m)` |
| Vectores por capa | `modelo(**entrada, output_hidden_states=True).hidden_states` |

## Referencias

- Bojanowski, P., Grave, E., Joulin, A. y Mikolov, T. (2017). Enriching word vectors with subword information. *TACL*, 5, 135–146.
- Bolukbasi, T., Chang, K.-W., Zou, J., Saligrama, V. y Kalai, A. (2016). Man is to computer programmer as woman is to homemaker? Debiasing word embeddings. *NeurIPS*.
- Cañete, J., Chaperon, G., Fuentes, R., Ho, J.-H., Kang, H. y Pérez, J. (2020). Spanish pre-trained BERT model and evaluation data. *PML4DC @ ICLR*.
- Devlin, J., Chang, M.-W., Lee, K. y Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL*.
- Firth, J. R. (1957). A synopsis of linguistic theory, 1930–1955. En *Studies in Linguistic Analysis*. Blackwell.
- Grave, E., Bojanowski, P., Gupta, P., Joulin, A. y Mikolov, T. (2018). Learning word vectors for 157 languages. *LREC*.
- Hamilton, W. L., Leskovec, J. y Jurafsky, D. (2016). Diachronic word embeddings reveal statistical laws of semantic change. *ACL*.
- Harris, Z. (1954). Distributional structure. *Word*, 10(2–3), 146–162.
- McCrae, J. P. (2025). *Linguistic Data Science* (worksheets). ESSLLI 2025.
- Mikolov, T., Chen, K., Corrado, G. y Dean, J. (2013). Efficient estimation of word representations in vector space. *arXiv:1301.3781*.
- Nissim, M., van Noord, R. y van der Goot, R. (2020). Fair is better than sensational: Man is to doctor as woman is to doctor. *Computational Linguistics*, 46(2), 487–497.